## PromptTemplate (최신 권장 방식)

> 원본: 테디노트 「랭체인을 활용한 RAG 비법노트」 CH02 `01-PromptTemplate.ipynb`  
> 기준: LangChain 1.x / langchain-core 1.2+ / LangSmith SDK (2026년 9월)

In [ ]:
# 필요 패키지 설치 (최초 1회)
# %pip install -qU langchain langchain-openai langsmith python-dotenv

### 🔄 변경 사항: 환경 설정
- `langchain_teddynote.logging.langsmith()` → LangSmith 공식 환경변수 `LANGSMITH_TRACING`, `LANGSMITH_PROJECT` 직접 설정  
  (써드파티 래퍼 없이 동일하게 동작합니다. `.env` 파일에 넣어 두어도 됩니다.)

In [ ]:
import os
from dotenv import load_dotenv

# .env 에 OPENAI_API_KEY, LANGSMITH_API_KEY 를 넣어 두세요.
load_dotenv()

# LangSmith 추적 설정
# (langchain_teddynote.logging.langsmith() 대신, LangSmith 공식 환경변수를 직접 설정)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "CH02-Prompt"

LLM 객체를 정의합니다.

### 🔄 변경 사항: LLM 객체 생성
- `ChatOpenAI()` → `init_chat_model("openai:...")` : LangChain v1 에서 권장하는 **공급자 중립적** 모델 초기화 방식입니다.  
  (`from langchain_openai import ChatOpenAI` 직접 사용도 여전히 유효합니다.)
- GPT-5 계열 모델은 `temperature` 값 지정에 제약이 있어 생략했습니다. `gpt-4.1` 등 이전 세대 모델을 쓸 때는 `init_chat_model(MODEL, temperature=0)` 처럼 전달하면 됩니다.
- 응답 텍스트는 `.content` 대신 **`.text`** 를 사용합니다. 모델/API(Responses API 등)에 따라 `.content` 가 문자열이 아닌 블록 리스트일 수 있기 때문입니다. 체인에서는 `StrOutputParser()` 를 붙이는 것이 가장 간단합니다.

In [ ]:
from langchain.chat_models import init_chat_model

# 모델은 "provider:model" 문자열 하나로 지정합니다.
# 다른 모델로 바꾸려면 이 한 줄만 수정하면 됩니다. (예: "anthropic:claude-sonnet-4-6")
MODEL = "openai:gpt-5.4-mini"

llm = init_chat_model(MODEL)

### 방법 1. `from_template()` 메소드를 사용하여 PromptTemplate 객체 생성

- 치환될 변수를 `{ 변수 }` 로 묶어서 템플릿을 정의합니다.

In [ ]:
from langchain_core.prompts import PromptTemplate

# template 정의. {country}는 변수로, 이후에 값이 들어갈 자리를 의미
template = "{country}의 수도는 어디인가요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)
prompt

`country` 변수에 값을 넣어서 문장을 생성할 수 있습니다.

### 🔄 변경 사항
- 원본은 `prompt = prompt.format(...)` 으로 **템플릿 객체를 문자열로 덮어써서** 이후 셀에서 템플릿을 다시 만들어야 했습니다. 결과는 별도 변수에 담습니다.
- `format()` 은 여전히 유효하지만, 모든 LangChain 구성요소가 공유하는 표준 인터페이스는 **`invoke()`** 입니다. `invoke()` 는 `PromptValue` 를 반환하며, 문자열/메시지 어느 쪽으로도 변환할 수 있습니다.

In [ ]:
# format(): 문자열 반환
formatted = prompt.format(country="대한민국")
formatted

In [ ]:
# invoke(): Runnable 표준 인터페이스 → PromptValue 반환
prompt_value = prompt.invoke({"country": "대한민국"})

print(prompt_value.to_string())    # 문자열 형태
print(prompt_value.to_messages())  # 채팅 메시지 형태

### 🔄 변경 사항: 체인 실행
- `chain.invoke("대한민국").content` → 입력은 **dict** 로 명시하고, 출력은 `StrOutputParser()` 로 문자열 변환  
  (변수가 하나일 때 문자열만 넘겨도 동작은 하지만, 변수가 늘어나면 깨지므로 dict 가 권장됩니다.)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# chain 생성
chain = prompt | llm | StrOutputParser()

In [ ]:
# country 변수에 입력된 값이 자동으로 치환되어 수행됨
chain.invoke({"country": "대한민국"})

### 방법 2. PromptTemplate 객체 생성과 동시에 prompt 생성

추가 유효성 검사를 위해 `input_variables` 를 명시적으로 지정할 수 있습니다.

### 🔄 변경 사항
- 현재 버전에서는 `input_variables` 를 지정해도 **기본적으로는 검증하지 않습니다.** 템플릿 변수와 불일치 시 예외를 발생시키려면 `validate_template=True` 를 함께 지정해야 합니다.
- 일반적으로는 변수를 자동 추론하는 `from_template()` 이 더 권장됩니다.

In [ ]:
# template 정의
template = "{country}의 수도는 어디인가요?"

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template=template,
    input_variables=["country"],
    validate_template=True,  # 불일치 시 예외 발생
)

prompt

In [ ]:
# prompt 생성
prompt.format(country="대한민국")

In [ ]:
# 검증 동작 확인: 템플릿 변수와 input_variables 가 다르면 오류
try:
    PromptTemplate(
        template="{country}의 수도는 어디인가요?",
        input_variables=["nation"],
        validate_template=True,
    )
except Exception as e:
    print(type(e).__name__, ":", e)

In [ ]:
# template 정의
template = "{country1}과 {country2}의 수도는 각각 어디인가요?"

# PromptTemplate 객체를 활용하여 prompt_template 생성
prompt = PromptTemplate(
    template=template,
    input_variables=["country1"],
    partial_variables={
        "country2": "미국"  # dictionary 형태로 partial_variables를 전달
    },
)

prompt

In [ ]:
prompt.format(country1="대한민국")

같은 결과를 `from_template()` + `.partial()` 조합으로 더 간결하게 만들 수 있습니다. (권장)

In [ ]:
prompt = PromptTemplate.from_template(template).partial(country2="미국")
prompt.format(country1="대한민국")

In [ ]:
prompt_partial = prompt.partial(country2="캐나다")
prompt_partial

In [ ]:
prompt_partial.format(country1="대한민국")

In [ ]:
chain = prompt_partial | llm | StrOutputParser()

In [ ]:
chain.invoke({"country1": "대한민국"})

In [ ]:
# partial 로 채운 값도 호출 시 덮어쓸 수 있습니다.
chain.invoke({"country1": "대한민국", "country2": "호주"})

### `partial_variables`: 부분 변수 채움

`partial`을 사용하는 일반적인 용도는 함수를 부분적으로 사용하는 것입니다. 이 사용 사례는 **항상 공통된 방식으로 가져오고 싶은 변수** 가 있는 경우입니다.

대표적인 예가 **날짜나 시간** 입니다.

항상 현재 날짜가 표시되기를 원하는 프롬프트가 있다고 가정해 보겠습니다. 이 경우 항상 현재 **날짜를 반환하는 함수** 를 사용하여 프롬프트를 부분적으로 변경할 수 있으면 매우 편리합니다. (이 기능은 현재 버전에서도 동일하게 동작합니다.)

다음의 코드는 오늘 날짜를 구하는 파이썬 코드입니다.

In [ ]:
from datetime import datetime

# 오늘 날짜를 출력
datetime.now().strftime("%B %d")

In [ ]:
# 날짜를 반환하는 함수 정의
def get_today():
    return datetime.now().strftime("%B %d")

In [ ]:
prompt = PromptTemplate.from_template(
    "오늘의 날짜는 {today} 입니다. 오늘이 생일인 유명인 {n}명을 나열해 주세요. 생년월일을 표기해주세요."
).partial(today=get_today)  # 함수 자체를 전달 → format/invoke 시점마다 호출됨

In [ ]:
# prompt 생성
prompt.format(n=3)

In [ ]:
# chain 을 생성합니다.
chain = prompt | llm | StrOutputParser()

In [ ]:
# chain 을 실행 후 결과를 확인합니다.
print(chain.invoke({"n": 3}))

In [ ]:
# partial 값을 덮어써서 실행
print(chain.invoke({"today": "Jan 02", "n": 3}))

## 파일로부터 template 읽어오기

### 🔄 변경 사항
- `langchain_core.prompts.load_prompt()` 와 `prompt.save()` 는 **langchain-core 1.2.21 부터 deprecated** 되었고 2.0 에서 제거될 예정입니다.
- 공식 대체 수단은 LangChain 표준 직렬화인 **`langchain_core.load` 의 `dumps` / `loads`** 입니다.
- Windows 인코딩 문제용 `langchain_teddynote.prompts.load_prompt` 도 필요 없습니다. 파일을 열 때 `encoding="utf-8"` 을 명시하면 됩니다.
- 프롬프트를 팀과 공유·버전 관리하려면 파일 대신 **LangSmith Prompt Hub** 를 쓰는 것이 권장됩니다. (`03-LangChain-Hub.ipynb` 참고)

### 방법 A (권장). `dumps` / `loads` 로 저장·불러오기

In [ ]:
from pathlib import Path
from langchain_core.load import dumps, loads

Path("prompts").mkdir(exist_ok=True)

# 저장: 프롬프트 객체 → JSON
fruit_prompt = PromptTemplate.from_template("{fruit}의 색깔이 뭐야?")
Path("prompts/fruit_color.json").write_text(
    dumps(fruit_prompt, pretty=True), encoding="utf-8"
)

In [ ]:
# 불러오기: JSON → 프롬프트 객체
# (기본 allowed_objects="core" 로 langchain_core 의 프롬프트/메시지 타입만 역직렬화 허용 → 안전)
prompt = loads(Path("prompts/fruit_color.json").read_text(encoding="utf-8"))
prompt

In [ ]:
prompt.format(fruit="사과")

### 방법 B. 책에서 사용한 기존 YAML 파일을 그대로 읽기

이미 가지고 있는 `prompts/*.yaml` 파일(`template` 키를 가진 형식)은 PyYAML 로 읽어서 `PromptTemplate` 을 직접 만들면 됩니다. deprecated API 에 의존하지 않습니다.

In [ ]:
import yaml


def load_prompt_yaml(path: str) -> PromptTemplate:
    """template 키를 가진 YAML 파일에서 PromptTemplate 생성"""
    with open(path, encoding="utf-8") as f:  # 인코딩을 명시 → Windows 에서도 동작
        config = yaml.safe_load(f)
    return PromptTemplate.from_template(config["template"])


prompt = load_prompt_yaml("prompts/fruit_color.yaml")
prompt.format(fruit="사과")

In [ ]:
prompt2 = load_prompt_yaml("prompts/capital.yaml")
print(prompt2.format(country="대한민국"))

## ChatPromptTemplate

`ChatPromptTemplate` 은 대화목록을 프롬프트로 주입하고자 할 때 활용할 수 있습니다.

메시지는 튜플(tuple) 형식으로 구성하며, (`role`, `message`) 로 구성하여 리스트로 생성할 수 있습니다.

**role**
- `"system"`: 시스템 설정 메시지 입니다. 주로 전역설정과 관련된 프롬프트입니다.
- `"human"` (= `"user"`) : 사용자 입력 메시지 입니다.
- `"ai"` (= `"assistant"`): AI 의 답변 메시지입니다.

### 🔄 변경 사항
- `ChatPromptTemplate.from_messages([...])` 는 여전히 유효하며, 생성자 `ChatPromptTemplate([...])` 로도 동일하게 만들 수 있습니다. (최신 공식 문서는 생성자 형태를 주로 사용)
- 채팅 프롬프트에 `.format()` 을 쓰면 `"Human: ..."` 형태의 **문자열**이 나옵니다. 채팅 모델에 넣을 때는 `.invoke()` 또는 `.format_messages()` 로 **메시지 리스트**를 만드는 것이 맞습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template("{country}의 수도는 어디인가요?")
chat_prompt

In [ ]:
# 문자열로 변환 (디버깅/확인용)
chat_prompt.format(country="대한민국")

In [ ]:
# 채팅 모델 입력용: 메시지 리스트
chat_prompt.invoke({"country": "대한민국"}).to_messages()

In [ ]:
chat_template = ChatPromptTemplate(
    [
        # role, message
        ("system", "당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name} 입니다."),
        ("human", "반가워요!"),
        ("ai", "안녕하세요! 무엇을 도와드릴까요?"),
        ("human", "{user_input}"),
    ]
)

# 챗 message 를 생성합니다.
messages = chat_template.format_messages(
    name="테디", user_input="당신의 이름은 무엇입니까?"
)
messages

생성한 메시지를 바로 주입하여 결과를 받을 수 있습니다.

In [ ]:
llm.invoke(messages).text

이번에는 체인을 생성해 보겠습니다.

In [ ]:
chain = chat_template | llm | StrOutputParser()

In [ ]:
chain.invoke({"name": "Teddy", "user_input": "당신의 이름은 무엇입니까?"})

## MessagesPlaceholder

LangChain은 포맷하는 동안 렌더링할 메시지를 완전히 제어할 수 있는 `MessagesPlaceholder` 를 제공합니다.

메시지 프롬프트 템플릿에 어떤 역할을 사용해야 할지 확실하지 않거나 서식 지정 중에 메시지 목록을 삽입하려는 경우 유용할 수 있습니다.

### 🔄 변경 사항
- 사용법은 동일합니다. 대화 목록은 튜플뿐 아니라 OpenAI 스타일 dict(`{"role": ..., "content": ...}`) 나 `HumanMessage`/`AIMessage` 객체로도 전달할 수 있습니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate(
    [
        (
            "system",
            "당신은 요약 전문 AI 어시스턴트입니다. 당신의 임무는 주요 키워드로 대화를 요약하는 것입니다.",
        ),
        MessagesPlaceholder(variable_name="conversation"),
        ("human", "지금까지의 대화를 {word_count} 단어로 요약합니다."),
    ]
)
chat_prompt

`conversation` 대화목록을 나중에 추가하고자 할 때 `MessagesPlaceholder` 를 사용할 수 있습니다.

In [ ]:
conversation = [
    {"role": "user", "content": "안녕하세요! 저는 오늘 새로 입사한 테디 입니다. 만나서 반갑습니다."},
    {"role": "assistant", "content": "반가워요! 앞으로 잘 부탁 드립니다."},
]

formatted_chat_prompt = chat_prompt.format(word_count=5, conversation=conversation)

print(formatted_chat_prompt)

In [ ]:
# chain 생성
chain = chat_prompt | llm | StrOutputParser()

In [ ]:
# chain 실행 및 결과확인
chain.invoke({"word_count": 5, "conversation": conversation})